# Nutrition CallBot — Embedding + Qdrant Ingest

Pipeline:
1. Mount Google Drive → đọc `nutrition_chunks.jsonl`
2. Embed bằng `intfloat/multilingual-e5-large-instruct` (1024-dim)
3. Upload lên Qdrant Cloud

**Yêu cầu trước khi chạy:**
- Runtime: GPU T4 (Runtime → Change runtime type → T4 GPU)
- Upload `nutrition_chunks.jsonl` lên Google Drive
- Điền QDRANT_URL và QDRANT_API_KEY ở Cell 3

In [ ]:
# Cell 1: Cài dependencies
!pip install -q sentence-transformers qdrant-client

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: CONFIG
QDRANT_URL     = "https://YOUR-CLUSTER-URL.aws.cloud.qdrant.io"
QDRANT_API_KEY = "YOUR-API-KEY"
COLLECTION     = "nutrition_articles"

DATA_PATH        = "/content/drive/MyDrive/Nutrition-Callbot/nutrition_chunks.jsonl"
MODEL_NAME       = "intfloat/multilingual-e5-large-instruct"
EMBED_BATCH_SIZE = 64    # giảm xuống 32 nếu OOM
MAX_CHUNKS       = None  # None = toàn bộ; đặt 200 để test nhanh trước

In [ ]:
# Cell 4: Load chunks
import json

print(f"Loading {DATA_PATH}...")
chunks = []
bad_lines = 0
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        try:
            chunks.append(json.loads(line))
        except json.JSONDecodeError as e:
            bad_lines += 1
            if bad_lines <= 3:
                print(f"  [WARN] Line {i+1} loi JSON: {e} | preview: {line[:80]}")

total = len(chunks)
if bad_lines:
    print(f"\n[WARN] {bad_lines} lines bi loi va da bo qua")
    print(f"=> File co the bi corrupt khi upload. Re-upload lai nutrition_chunks.jsonl")

if MAX_CHUNKS:
    chunks = chunks[:MAX_CHUNKS]
    print(f"Loaded {total:,} chunks -> gioi han test: {len(chunks):,}")
else:
    print(f"Loaded {total:,} chunks")

sources = {}
for c in chunks:
    sources[c["source"]] = sources.get(c["source"], 0) + 1
print("\nPhan bo:")
for src, cnt in sorted(sources.items()):
    print(f"  {src}: {cnt:,} chunks")

c0 = chunks[0]
print(f"\nVi du chunk dau tien:")
print(f"  chunk_id : {c0['chunk_id']}")
print(f"  title    : {c0['title'][:80]}")
print(f"  text     : {c0['text'][:150]}...")

In [ ]:
# Cell 5: Load embedding model
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

print(f"\nLoading {MODEL_NAME}... (lan dau ~3 phut)")
model = SentenceTransformer(MODEL_NAME, device=device)

DIM = model.get_sentence_embedding_dimension()
print(f"Model loaded! Dim: {DIM}")

test = model.encode(["passage: Dinh duong la gi?"], normalize_embeddings=True)
print(f"Test vector shape: {test.shape}")

In [ ]:
# Cell 6: Ket noi Qdrant + tao collection
from qdrant_client import QdrantClient
from qdrant_client.http import models
import hashlib, uuid

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)
info = qdrant.get_collections()
print(f"Ket noi OK! Collections hien co: {[c.name for c in info.collections]}")

if qdrant.collection_exists(COLLECTION):
    print(f"Xoa collection cu: {COLLECTION}")
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE),
)

for field in ["source", "category", "doc_id"]:
    qdrant.create_payload_index(COLLECTION, field, field_schema="keyword")

print(f"Tao collection '{COLLECTION}' (dim={DIM}) thanh cong")

In [ ]:
# Cell 7: Embed + Upload
import time

def make_uuid(chunk_id: str) -> str:
    h = hashlib.md5(chunk_id.encode()).hexdigest()
    return str(uuid.UUID(h))


def embed_and_upload(chunks_list, batch_size=EMBED_BATCH_SIZE):
    total = len(chunks_list)
    uploaded = 0
    t_start = time.time()

    for batch_start in range(0, total, batch_size):
        batch = chunks_list[batch_start : batch_start + batch_size]

        # Prefix "passage:" theo chuan E5 — embed_text = title + content
        texts = [f"passage: {c['embed_text']}" for c in batch]
        vecs = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

        points = [
            models.PointStruct(
                id=make_uuid(c["chunk_id"]),
                vector=vecs[i].tolist(),
                payload={
                    "chunk_id"   : c["chunk_id"],
                    "doc_id"     : c["doc_id"],
                    "source"     : c["source"],
                    "url"        : c["url"],
                    "title"      : c["title"],
                    "category"   : c["category"],
                    "chunk_index": c["chunk_index"],
                    "text"       : c["text"],
                },
            )
            for i, c in enumerate(batch)
        ]

        qdrant.upsert(collection_name=COLLECTION, points=points)
        uploaded += len(batch)

        elapsed = time.time() - t_start
        speed = uploaded / elapsed
        eta = (total - uploaded) / speed if speed > 0 else 0
        if uploaded % (batch_size * 20) == 0 or uploaded == total:
            print(
                f"  {uploaded:,}/{total:,} ({uploaded/total*100:.1f}%) | "
                f"{speed:.0f} chunks/s | ETA: {eta/60:.1f} min"
            )

    print(f"\nHoan tat! {uploaded:,} vectors | {(time.time()-t_start)/60:.1f} phut")


print(f"Bat dau embedding + upload {len(chunks):,} chunks...")
embed_and_upload(chunks)

col_info = qdrant.get_collection(COLLECTION)
print(f"\nQdrant '{COLLECTION}': {col_info.points_count:,} vectors da luu")

In [ ]:
# Cell 8: Test search

def search(query: str, top_k: int = 3, filter_source: str = None):
    q_vec = model.encode(
        [f"Instruct: Tim thong tin dinh duong lien quan\nQuery: {query}"],
        normalize_embeddings=True,
    )[0].tolist()

    search_filter = None
    if filter_source:
        search_filter = models.Filter(
            must=[models.FieldCondition(key="source", match=models.MatchValue(value=filter_source))]
        )

    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=q_vec,
        limit=top_k,
        query_filter=search_filter,
        with_payload=True,
    )
    return results.points


test_queries = [
    "Tre em may thang tuoi thi bat dau an dam?",
    "Nguoi tieu duong nen an gi va kieng gi?",
    "Thieu canxi co bieu hien nhu the nao?",
]

for q in test_queries:
    print(f"\nQuery: '{q}'")
    hits = search(q)
    for i, p in enumerate(hits):
        print(f"  {i+1}. [{p.score:.4f}] [{p.payload['source']}] {p.payload['title'][:70]}")
        print(f"       {p.payload['text'][:130]}...")